# 第一章：亲手走完一次文本生成

这一章只回答一个问题：**一段输入文字，怎样变成逐个输出的 token？**

先运行一次计算，看见结果，再把同样的步骤放进循环。每个代码格按 **Shift + Enter** 运行。
首次使用先看[启动说明](../README.md)。从上到下运行；改了前面的设置后，用 **Restart Kernel and Run All Cells** 重新开始。

```text
文字 → tokenizer → 输入 IDs → 模型 → logits → argmax → 新 ID
                       ↑                              │
                       └──────── 接回输入 ────────────┘
最后：生成的 IDs → tokenizer.decode → 输出文字
```

本章沿用 `tutorial-01` 的 HF 正确性 baseline：Qwen3.5-0.8B、FP32、无 cache、greedy。
HF 负责模型内部计算，我们写生成循环。后续章节才逐步替换模型内部、加 cache 和调度。
这里先看清过程，不测性能。

## 1. 准备：我们正在用哪个 Python？

先打印版本。这样遇到问题时，知道 Notebook 和终端是否使用了同一套环境。
这里的 `torch` 负责张量运算；`transformers` 提供本章借用的模型和 tokenizer。

In [1]:
import gc
import json
import sys
from importlib.metadata import version
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

print('Python:', sys.version.split()[0])
for package in ['torch', 'transformers', 'huggingface_hub']:
    print(package + ':', version(package))
print('解释器:', sys.executable)

Python: 3.13.14
torch: 2.14.1
transformers: 5.19.0
huggingface_hub: 2.2.0
解释器: /Users/ryanli/.cache/uv/builds-v0/.tmpysgUIH/bin/python


本次只生成 8 个 token，答案可能还没说完。`mps` 表示使用 Mac 的 Apple GPU；没有 MPS 时自动使用 CPU。
权重固定到同一个 revision，避免模型文件更新后影响对照。

In [2]:
PROMPT = '用一句话解释什么是 KV cache。'
MAX_NEW_TOKENS = 8
MODEL_ID = 'Qwen/Qwen3.5-0.8B'
MODEL_REVISION = '2fc06364715b967f1860aea9cf38778875588b17'
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('设备:', DEVICE)

设备: mps


## 2. 文字怎样变成数字？

先只加载 tokenizer。下面从本地读取已经下载好的模型文件；没有缓存时，按[启动说明](../README.md)下载一次。

In [3]:
model_dir = snapshot_download(
    MODEL_ID, revision=MODEL_REVISION, local_files_only=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
small_ids = tokenizer.encode('你好', add_special_tokens=False)
print('文字 → IDs:', small_ids)
print('IDs → 文字:', tokenizer.decode(small_ids))

文字 → IDs: [109266]
IDs → 文字: 你好


token 是模型使用的文本片段，不保证恰好是一个汉字或一个单词。

聊天模型还需要知道“谁在说话”和“接下来该谁回答”。先查看 **chat template** 组织出来的文本。
这一步只用于观察；真正编码时直接用下一格的 `tokenize=True`，避免自行拼接、重复添加特殊标记。

In [4]:
messages = [{'role': 'user', 'content': PROMPT}]
chat_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
)
print(chat_text)

<|im_start|>user
用一句话解释什么是 KV cache。<|im_end|>
<|im_start|>assistant
<think>

</think>




`add_generation_prompt=True` 放入回答开头需要的标记；`enable_thinking=False` 沿用本章 baseline 的模板设置。
接下来得到真正送入模型的二维整数张量。

In [5]:
encoded = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, enable_thinking=False,
    return_dict=True, return_tensors='pt',
)
prompt_input_ids = encoded['input_ids'].to(DEVICE)
prompt_ids = prompt_input_ids[0].tolist()
print('shape:', tuple(prompt_input_ids.shape))
print('IDs:', prompt_ids)

shape: (1, 19)
IDs: [248045, 846, 198, 95826, 110827, 98682, 108829, 81726, 6297, 1710, 248046, 198, 248045, 74455, 198, 248068, 271, 248069, 271]


第一维是输入条数，当前为 1；第二维是这条输入的 token 数。默认问题的历史记录是 `(1, 19)`。
这些 19 个 token 包含聊天标记，不只是问题正文。看下面的对应关系，找找哪些是特殊标记。
单个 token 的字符串可能带编码符号，因此最终输出会整串 decode。

In [6]:
print('位置 | token ID | tokenizer 内部片段')
for position, token_id in enumerate(prompt_ids[:32]):
    print(
        f'{position:>4} | {token_id:>8} | '
        f'{tokenizer.convert_ids_to_tokens(token_id)!r}'
    )
if len(prompt_ids) > 32:
    print(f'其余 {len(prompt_ids) - 32} 个 token 省略显示。')

位置 | token ID | tokenizer 内部片段
   0 |   248045 | '<|im_start|>'
   1 |      846 | 'user'
   2 |      198 | 'Ċ'
   3 |    95826 | 'çĶ¨'
   4 |   110827 | 'ä¸Ģåı¥è¯Ŀ'
   5 |    98682 | 'è§£éĩĬ'
   6 |   108829 | 'ä»Ģä¹Īæĺ¯'
   7 |    81726 | 'ĠKV'
   8 |     6297 | 'Ġcache'
   9 |     1710 | 'ãĢĤ'
  10 |   248046 | '<|im_end|>'
  11 |      198 | 'Ċ'
  12 |   248045 | '<|im_start|>'
  13 |    74455 | 'assistant'
  14 |      198 | 'Ċ'
  15 |   248068 | '<think>'
  16 |      271 | 'ĊĊ'
  17 |   248069 | '</think>'
  18 |      271 | 'ĊĊ'


## 3. 模型一次返回什么？

现在加载模型。`float32` 是本章的数值精度；`.eval()` 切换到推理模式。
加载时出现 FLA / causal_conv1d 的 fallback 提示是这条 HF 路径的预期情况：使用通用 PyTorch 运算。

模型会占用数 GB 内存。重复运行这一格前会释放旧对象；本章末尾也提供释放步骤。

In [7]:
if 'model' in globals():
    del model
gc.collect()
if DEVICE == 'mps':
    torch.mps.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    model_dir, local_files_only=True, dtype=torch.float32,
).to(DEVICE).eval()
print('模型参数精度:', next(model.parameters()).dtype)

模型参数精度: torch.float32


**关键①：调用模型计算分数。**
先只调用一次，输入仍是原问题。`inference_mode()` 关闭训练需要的梯度记录；`use_cache=False` 不保留用于加速后续生成的状态。

In [8]:
with torch.inference_mode():
    logits = model(input_ids=prompt_input_ids, use_cache=False).logits

print('输入 shape:', tuple(prompt_input_ids.shape))
print('logits shape:', tuple(logits.shape))
assert logits.shape[:2] == prompt_input_ids.shape

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


输入 shape: (1, 19)
logits shape: (1, 19, 248320)


logits 是候选 token 的分数，形状为 `[B, T, V]`：输入条数、输入长度、词表大小。
它还不是概率。

```text
输入位置：       A          B          C
预测依据：      [A]       [A,B]      [A,B,C]
输出：        V个分数    V个分数     V个分数 ← 生成时选这一组
```

我们想接在整个问题后面继续写，所以取最后一个位置的分数：`logits[0, -1]`。

In [9]:
next_scores = logits[0, -1].detach().cpu().clone()
print('最后一个位置的分数 shape:', tuple(next_scores.shape))
print('候选 token 数:', next_scores.numel())
del logits  # 只留这一行 CPU 分数，释放完整输出

最后一个位置的分数 shape: (248320,)
候选 token 数: 248320


## 4. 从分数里选哪个 token？

先用一个假设的小词表，只看懂 `argmax`：它返回最大值的**下标**。
在真实 logits 里，下标就是 token ID。

In [10]:
toy_tokens = ['甲', '乙', '丙']
toy_scores = torch.tensor([1.2, 3.5, 0.8])
toy_id = int(toy_scores.argmax().item())
print('argmax 下标:', toy_id)
print('对应 token:', toy_tokens[toy_id])
print('先 softmax 再 argmax:', int(toy_scores.softmax(dim=0).argmax().item()))

argmax 下标: 1
对应 token: 乙
先 softmax 再 argmax: 1


每次直接选最高分叫 **greedy**。softmax 保持分数的大小顺序，因此这里只为选最高分，不需要先算概率。
换成模型的真实分数：

In [11]:
first_id = int(next_scores.argmax().item())  # ← 关键②：选一个 ID
top_scores, top_ids = torch.topk(next_scores, k=5)
print('候选 ID | 分数（非概率） | 单个 token 解码预览')
for token_id, score in zip(top_ids.tolist(), top_scores.tolist()):
    print(f'{token_id:>7} | {score:>14.4f} | {tokenizer.decode([token_id])!r}')
print('选中:', first_id)

候选 ID | 分数（非概率） | 单个 token 解码预览
  79852 |        24.0883 | 'KV'
     42 |        19.1351 | 'K'
    332 |        18.5010 | '**'
  97018 |        17.7526 | '为了'
  96540 |        17.4754 | '它'
选中: 79852


## 5. 有了一个新 ID，怎样继续？

**关键③：把新 ID 接回输入。** 请先猜：第二次调用模型时，输入长度增加多少？
这里使用一个新变量做演示，保留原始 `prompt_input_ids`，方便重复实验。

In [12]:
next_input_ids = torch.cat([
    prompt_input_ids,
    torch.tensor([[first_id]], device=DEVICE),
], dim=1)
print('原输入长度:', prompt_input_ids.shape[1])
print('下一轮输入长度:', next_input_ids.shape[1])
print('下一轮末尾 ID:', int(next_input_ids[0, -1].item()))
assert next_input_ids.shape[1] == prompt_input_ids.shape[1] + 1

原输入长度: 19
下一轮输入长度: 20
下一轮末尾 ID: 79852


刚选出的 ID 只是预测结果，还没有作为输入经过模型。接回去后，模型才会以它为已知条件继续预测。

```text
第1轮：[A,B,C]     → X
第2轮：[A,B,C,X]   → Y
第3轮：[A,B,C,X,Y] → Z
```

## 6. 把刚才三步组成循环

什么时候停？最多生成 `MAX_NEW_TOKENS` 个，或者遇到 EOS（结束标记）提前停。
EOS 可能不止一个，沿用模型配置中的完整集合。

In [13]:
configured_eos = model.generation_config.eos_token_id
eos_ids = set(configured_eos if isinstance(configured_eos, list) else [configured_eos])
print('结束标记 IDs:', sorted(eos_ids))

结束标记 IDs: [248044]


下面没有引入新的生成算法，只是重复刚才三步。每次运行都从原问题的副本开始，
`generated` 和 `rounds` 也重新建立，不会接着上次输出继续写。

In [14]:
assert MAX_NEW_TOKENS >= 1
input_ids = prompt_input_ids.clone()
generated, rounds = [], []
with torch.inference_mode():
    for step in range(MAX_NEW_TOKENS):
        logits = model(input_ids=input_ids, use_cache=False).logits  # ① 计算分数
        next_id = int(logits[0, -1].argmax().item())                 # ② 选 ID
        rounds.append({
            'round': step + 1, 'input_tokens': input_ids.shape[1],
            'logits_shape': list(logits.shape), 'next_token_id': next_id,
        })
        generated.append(next_id)
        input_ids = torch.cat([                                    # ③ 接回输入
            input_ids, torch.tensor([[next_id]], device=DEVICE),
        ], dim=1)
        del logits
        if next_id in eos_ids:
            break

逐轮检查：输入长度和 logits 的 T 维是否每次加 1？这就是 `use_cache=False` 带来的整段重算。

In [15]:
print('轮次 | 输入长度 | logits shape | 新 ID')
for row in rounds[:32]:
    print(
        row['round'], row['input_tokens'],
        row['logits_shape'], row['next_token_id'], sep=' | ',
    )
print('生成 IDs:', generated[:32])
print('输出文字:', tokenizer.decode(generated, skip_special_tokens=True))
print('停止原因:', 'EOS' if generated[-1] in eos_ids else '达到 token 上限')

轮次 | 输入长度 | logits shape | 新 ID
1 | 19 | [1, 19, 248320] | 79852
2 | 20 | [1, 20, 248320] | 18887
3 | 21 | [1, 21, 248320] | 220
4 | 22 | [1, 22, 248320] | 98435
5 | 23 | [1, 23, 248320] | 98618
6 | 24 | [1, 24, 248320] | 103806
7 | 25 | [1, 25, 248320] | 95779
8 | 26 | [1, 26, 248320] | 103725
生成 IDs: [79852, 18887, 220, 98435, 98618, 103806, 95779, 103725]
输出文字: KV Cache 是一种用于加速大模型
停止原因: 达到 token 上限


第一轮处理已有问题并预测第一个输出，叫 **prefill**；随后继续生成叫 **decode**。
这份 baseline 的 decode 仍然重算整个前缀。后面的 cache 章节才会减少这项重复工作。

8 个 token 很短，不保证是一句完整的话。EOS 也是生成的 ID，但 `skip_special_tokens=True` 解码时会隐藏它。

## 7. 核对我们拆开的代码

先检查生成过程本身。第一轮应该与刚才单步演示一致；如果提前停止，原因必须是 EOS。

In [16]:
assert 1 <= len(generated) <= MAX_NEW_TOKENS
assert generated[0] == first_id
assert input_ids[0].tolist() == prompt_ids + generated
assert all(row['input_tokens'] == len(prompt_ids) + i for i, row in enumerate(rounds))
assert all(row['logits_shape'] == [1, len(prompt_ids) + i, next_scores.numel()]
           for i, row in enumerate(rounds))
assert not any(token_id in eos_ids for token_id in generated[:-1])
assert len(generated) == MAX_NEW_TOKENS or generated[-1] in eos_ids
print('形状、输入拼接、单步结果和停止条件：通过')

形状、输入拼接、单步结果和停止条件：通过


再与 `tutorial-01` 保存的历史记录对照。[原脚本](reference/generate.py)和
[历史 trace](reference/chapter01-trace.json)是从固定 commit 原样复制的，来源及哈希见
[manifest](reference/manifest.json)。Notebook 是新的教学拆解，旧 tag 保持不变。

以下会打印历史 ID 对照是否一致。更换 prompt 后跳过这项；更换硬件或依赖后若出现差异，
需要检查分数和数值路径，不能直接把历史结果改成新结果。

In [17]:
# 兼容从项目根目录或本章目录启动 Jupyter。
here = Path.cwd().resolve()
chapter_dir = next((p for p in [here, here / 'ch01', here / 'notebooks/ch01']
                    if (p / 'reference/manifest.json').is_file()), None)
assert chapter_dir is not None, '请从仓库根目录启动，或把 kernel 工作目录设为 notebooks/ch01。'
historical = json.loads((chapter_dir / 'reference/chapter01-trace.json').read_text())
if PROMPT == historical['prompt']:
    assert prompt_ids == historical['prompt_ids'], 'tokenizer 输入与历史记录不同'
    n = min(len(generated), len(historical['generated_ids']))
    historical_match = generated[:n] == historical['generated_ids'][:n]
    print(f'历史前 {n} 个输出 ID 一致:', historical_match)
else:
    print('已更换问题，跳过历史输出对照。')

历史前 8 个输出 ID 一致: True


## 8. 自己改一次

先预测结果，再动手。答案放在[独立练习本](exercise-solutions.ipynb)，可以晚一点打开。

1. 把小词表分数改成 `[4.0, 3.5, 0.8]`，greedy 选谁？为什么不是返回 `4.0`？
2. 默认问题有 19 个 token。第 8 轮输入长度是多少？注意刚选出的第 8 个 ID 尚未进入这一轮模型计算。
3. 把顶部 `MAX_NEW_TOKENS` 改成 12，重启并运行全部。前 8 个 ID 是否与之前相同？

## 9. 本章做了什么，下一章换什么？

| 位置 | 第一章 | 第二章 |
|---|---|---|
| tokenizer / chat template | 借用 HF | 保留 |
| 完整模型 forward | 借用 HF | 仍保留；先独立核对手写基础组件 |
| 选 token、拼输入、控制停止 | 我们自己写 | 保留 |
| 内部 Embedding / Linear 等 | 在 HF 模型里面 | 拆出来手写，并与参考逐项比较 |

所以第二章不会立即把完整模型换掉。先把组件写对，第五章再接成完整的手写 forward。
继续阅读[第二章](../../docs/tutorial-v2/02-从ID到向量再到分数.md)。

学完运行下面一格释放模型；要再次生成，需要重新运行模型加载格及后续步骤。
也可用 Jupyter 的 **Shut Down Kernel** 结束本章进程。只关闭浏览器标签不等于停止 kernel。

In [18]:
if 'model' in globals():
    del model
gc.collect()
if DEVICE == 'mps':
    torch.mps.empty_cache()
print('已释放模型。')

已释放模型。


### 编排参考与版本

参考 Sebastian Raschka 的 [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch)：
[ch02](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb) 的短代码与输出穿插、
[ch03](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch03/01_main-chapter-code/ch03.ipynb) 的先拆步再组合、独立练习答案。
这里的中文讲解、例子与图示围绕我们自己的 inference engine 重写。

本章 baseline commit：`4f9952da4537a2503477365b789a276e3ff8033d`（`tutorial-01`）。
模型 revision：`2fc06364715b967f1860aea9cf38778875588b17`。